# Discrete Feature Engineering and Weight of Evidence

This notebook investigates the original categorical predictors on the training partition, then applies the same category definitions to the held-out partition.

## Inputs

The prior notebook supplies the cleaned split and historical `good_bad` target. All WoE estimates and grouping decisions are learned from training data only.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

pd.set_option("display.max_rows", 100)
EPSILON = 1e-6
PROCESSED = Path("../data/processed")

In [ ]:
inputs_train = pd.read_pickle(PROCESSED / "clean_inputs_train.pkl")
inputs_test = pd.read_pickle(PROCESSED / "clean_inputs_test.pkl")
target_train = pd.read_pickle(PROCESSED / "targets_train.pkl")
target_test = pd.read_pickle(PROCESSED / "targets_test.pkl")
inputs_train.shape, inputs_test.shape, target_train.value_counts(normalize=True).rename("share")

## WoE and IV

WoE compares the distribution of good and bad observations across a category. IV summarizes the total separation for a feature. A category with no goods or no bads is clipped to `EPSILON = 1e-6` before the logarithm so the diagnostic remains finite and visible.

In [ ]:
def _woe_table(inputs: pd.DataFrame, feature: str, target: pd.Series, ordered: bool) -> pd.DataFrame:
    """Build a WoE table; zero good/bad distributions are clipped to EPSILON before log."""
    frame = pd.concat([inputs[feature], target.rename("good_bad")], axis=1)
    table = frame.groupby(feature, dropna=False, observed=False)["good_bad"].agg(n_obs="count", prop_good="mean").reset_index()
    if ordered:
        table = table.sort_values(feature, key=lambda values: values.astype(str)).reset_index(drop=True)
    table["prop_n_obs"] = table["n_obs"] / table["n_obs"].sum()
    table["n_good"] = table["n_obs"] * table["prop_good"]
    table["n_bad"] = table["n_obs"] - table["n_good"]
    table["prop_n_good"] = table["n_good"] / table["n_good"].sum()
    table["prop_n_bad"] = table["n_bad"] / table["n_bad"].sum()
    good_for_log = table["prop_n_good"].clip(lower=EPSILON)
    bad_for_log = table["prop_n_bad"].clip(lower=EPSILON)
    table["WoE"] = np.log(good_for_log / bad_for_log)
    table["diff_prop_good"] = table["prop_good"].diff().abs()
    table["diff_WoE"] = table["WoE"].diff().abs()
    table["IV"] = ((good_for_log - bad_for_log) * table["WoE"]).sum()
    return table


def woe_discrete(inputs: pd.DataFrame, feature: str, target: pd.Series) -> pd.DataFrame:
    """Return category counts, good/bad distributions, WoE, and feature IV."""
    return _woe_table(inputs, feature, target, ordered=False).sort_values("WoE").reset_index(drop=True)


def woe_ordered_continuous(inputs: pd.DataFrame, feature: str, target: pd.Series) -> pd.DataFrame:
    """Return ordered-bin counts, good/bad distributions, WoE, and feature IV."""
    return _woe_table(inputs, feature, target, ordered=True)


def plot_by_woe(table: pd.DataFrame, rotation: int = 0) -> None:
    """Plot WoE in table order with readable labels."""
    plt.figure(figsize=(18, 6))
    plt.plot(table.iloc[:, 0].astype(str), table["WoE"], marker="o", linestyle="--", color="black")
    plt.xlabel(table.columns[0])
    plt.ylabel("Weight of Evidence")
    plt.title(f"Weight of Evidence by {table.columns[0]}")
    plt.xticks(rotation=rotation)
    plt.show()


def add_groups(frame: pd.DataFrame, feature: str, groups: dict[str, list[str]]) -> pd.DataFrame:
    result = frame.copy()
    values = result[feature].fillna("Missing")
    for label, members in groups.items():
        result[f"{feature}:{label}"] = values.isin(members).astype(int)
    return result

## Manual construction for grade

The first table follows the original step-by-step calculation before the reusable helper is used for the remaining features.

In [ ]:
grade_manual = pd.concat([inputs_train["grade"], target_train.rename("good_bad")], axis=1)
grade_manual = grade_manual.groupby("grade", dropna=False)["good_bad"].agg(n_obs="count", prop_good="mean").reset_index()
grade_manual["prop_n_obs"] = grade_manual["n_obs"] / grade_manual["n_obs"].sum()
grade_manual["n_good"] = grade_manual["n_obs"] * grade_manual["prop_good"]
grade_manual["n_bad"] = grade_manual["n_obs"] - grade_manual["n_good"]
grade_manual["prop_n_good"] = grade_manual["n_good"] / grade_manual["n_good"].sum()
grade_manual["prop_n_bad"] = grade_manual["n_bad"] / grade_manual["n_bad"].sum()
grade_manual["WoE"] = np.log(grade_manual["prop_n_good"].clip(lower=EPSILON) / grade_manual["prop_n_bad"].clip(lower=EPSILON))
grade_manual["IV"] = ((grade_manual["prop_n_good"].clip(lower=EPSILON) - grade_manual["prop_n_bad"].clip(lower=EPSILON)) * grade_manual["WoE"]).sum()
grade_manual

In [ ]:
grade_woe = woe_discrete(inputs_train, "grade", target_train)
plot_by_woe(grade_woe)
grade_woe

## Categorical WoE diagnostics

Each table and plot is retained because it supports a category decision. Narrow range views keep the state distribution legible without discarding the full plot.

## Grade

The manual calculation is reproduced above. Grade remains a separate ordered category family so later model selection can choose an explicit reference.

In [ ]:
grade_woe = woe_discrete(inputs_train, "grade", target_train)
plot_by_woe(grade_woe)
grade_woe

## Home ownership

The original analysis combines RENT, OTHER, NONE, and ANY as a reference candidate while retaining OWN and MORTGAGE separately.

In [ ]:
home_ownership_woe = woe_discrete(inputs_train, "home_ownership", target_train)
plot_by_woe(home_ownership_woe)
home_ownership_woe

## Address state

The full and range-focused plots support the original state bands. The first low-risk/low-volume band is retained as the reference candidate.

In [ ]:
addr_state_woe = woe_discrete(inputs_train, "addr_state", target_train)
plot_by_woe(addr_state_woe, rotation=90)
addr_state_woe

## Verification status

Verification status remains an investigated categorical family; its IV is retained in the summary rather than assuming a strong contribution.

In [ ]:
verification_status_woe = woe_discrete(inputs_train, "verification_status", target_train)
plot_by_woe(verification_status_woe)
verification_status_woe

## Purpose

Smaller purposes are combined into the original reference candidate, while debt consolidation and credit card remain distinct because they are central lending purposes.

In [ ]:
purpose_woe = woe_discrete(inputs_train, "purpose", target_train)
plot_by_woe(purpose_woe, rotation=90)
purpose_woe

## Initial list status

Initial list status is retained as a separate diagnostic and candidate family for later model selection.

In [ ]:
initial_list_status_woe = woe_discrete(inputs_train, "initial_list_status", target_train)
plot_by_woe(initial_list_status_woe)
initial_list_status_woe

In [ ]:
plot_by_woe(discrete_woe["addr_state"].iloc[1:-2], rotation=90)
plot_by_woe(discrete_woe["addr_state"].iloc[6:-6], rotation=90)

## Train-derived category groups and references

`home_ownership:RENT_OTHER_NONE_ANY`, the first state group, and the first purpose group are reference-category candidates. The original group definitions are retained; they are evaluated on the training partition and applied unchanged to test rows.

In [ ]:
discrete_groups = {
    "grade": {grade: [grade] for grade in ["A", "B", "C", "D", "E", "F", "G"]},
    "home_ownership": {"RENT_OTHER_NONE_ANY": ["RENT", "OTHER", "NONE", "ANY"], "OWN": ["OWN"], "MORTGAGE": ["MORTGAGE"]},
    "addr_state": {
        "ND_NE_IA_NV_FL_HI_AL": ["ND", "NE", "IA", "NV", "FL", "HI", "AL"], "NM_VA": ["NM", "VA"],
        "OK_TN_MO_LA_MD_NC": ["OK", "TN", "MO", "LA", "MD", "NC"], "UT_KY_AZ_NJ": ["UT", "KY", "AZ", "NJ"],
        "AR_MI_PA_OH_MN": ["AR", "MI", "PA", "OH", "MN"], "RI_MA_DE_SD_IN": ["RI", "MA", "DE", "SD", "IN"],
        "GA_WA_OR": ["GA", "WA", "OR"], "WI_MT": ["WI", "MT"], "IL_CT": ["IL", "CT"],
        "KS_SC_CO_VT_AK_MS": ["KS", "SC", "CO", "VT", "AK", "MS"], "WV_NH_WY_DC_ME_ID": ["WV", "NH", "WY", "DC", "ME", "ID"]},
    "verification_status": {value: [value] for value in sorted(inputs_train["verification_status"].dropna().unique())},
    "purpose": {"educ__sm_b__wedd__ren_en__mov__house": ["educational", "small_business", "wedding", "renewable_energy", "moving", "house"], "oth__med__vacation": ["other", "medical", "vacation"], "major_purch__car__home_impr": ["major_purchase", "car", "home_improvement"], "debt_consolidation": ["debt_consolidation"], "credit_card": ["credit_card"]},
    "initial_list_status": {value: [value] for value in sorted(inputs_train["initial_list_status"].dropna().unique())},
}

In [ ]:
discrete_inputs_train = inputs_train.copy()
discrete_inputs_test = inputs_test.copy()
for feature, groups in discrete_groups.items():
    discrete_inputs_train = add_groups(discrete_inputs_train, feature, groups)
    discrete_inputs_test = add_groups(discrete_inputs_test, feature, groups)

created_discrete_columns = [column for column in discrete_inputs_train if ":" in column and column.split(":", 1)[0] in discrete_groups]
len(created_discrete_columns), created_discrete_columns

The `purpose` analysis keeps debt consolidation and credit card distinct, while smaller purposes are combined. State grouping follows the original WoE-informed bands. Verification status and initial listing status remain separate categories; their IV values show their empirical contribution rather than assuming a strong effect.

In [ ]:
discrete_iv_summary = pd.DataFrame({"feature": feature, "IV": table["IV"].iat[0]} for feature, table in discrete_woe.items()).sort_values("IV", ascending=False)
discrete_iv_summary

## Checkpoint and conclusions

These checkpoints retain the cleaned fields plus the discrete feature families for the next notebook. The held-out rows receive exactly the train-defined categories; no test distribution is used to choose a group or reference.

In [ ]:
discrete_inputs_train.to_pickle(PROCESSED / "discrete_inputs_train.pkl")
discrete_inputs_test.to_pickle(PROCESSED / "discrete_inputs_test.pkl")
print("Saved discrete checkpoints:", discrete_inputs_train.shape, discrete_inputs_test.shape)

**Conclusion.** Grade, home ownership, geography, verification status, purpose, and initial listing status are all explicitly investigated. IV is used as a descriptive ranking, not as a sole selection rule; the grouped families preserve the original reference logic and retain weak variables for the subsequent model-selection step.